In [1]:
import torch

In [2]:
# 1 a simple function
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [3]:
# Can also create x = torch.arange(4.0, requires_grad=True)
x.requires_grad_(True)
x.grad # The gradient is None by default

In [9]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [10]:
# Compute gradients
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [11]:
# reset gradients
x.grad.zero_() # Reset the gradient
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

### Backward for Non-Scalar Variables

In [13]:
# 1. Clear previous gradients. 
# PyTorch accumulates gradients by default; if we don't zero them out, 
# the new results will be added to whatever was stored in x.grad previously.
if x.grad is not None:
    x.grad.zero_()

# 2. Define the Forward Pass.
# We are creating the function y = f(x) = x^2.
y = x * x

print(y)

# 3. Perform the Backward Pass (Backpropagation).
# Since 'y' is a vector (tensor) and not a single scalar (like a loss value), 
# we must provide a 'gradient' argument of the same shape. 
# Passing torch.ones() essentially means dL/dy = 1 for each element.
# Mathematically, this computes dy/dx = 2*x.
y.backward(gradient=torch.ones(len(y))) 

# Note: The commented alternative 'y.sum().backward()' is more common. 
# It turns the vector into a scalar first, which simplifies the math 
# to the same result.

# 4. Access the result.
# x.grad now holds the values of the derivative evaluated at the points in x.
x.grad

tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)


tensor([0., 2., 4., 6.])

In [55]:
# Reset the gradients for the tensor x to zero to prevent accumulation from previous steps
x.grad.zero_()

# Compute y = x^2; y is part of the computational graph and tracks gradients
y = x * x

# Create a new tensor 'u' that has the same value as 'y' but is "detached" 
# from the gradient history. PyTorch won't backpropagate through u to x.
u = y.detach()

# Compute z = u * x. Since u is treated as a constant, 
# the derivative dz/dx is simply u.
z = u * x

# Sum all elements and compute gradients. 
# Backpropagation starts here, but stops at 'u'.
z.sum().backward()

# Verification: 
# x.grad should be equal to u because z = u * x (treating u as a constant).
# This returns (x, True, u, x.grad) assuming the values match.
x, x.grad == u, u, x.grad


(tensor([0., 1., 2., 3.], requires_grad=True),
 tensor([True, True, True, True]),
 tensor([0., 1., 4., 9.]),
 tensor([0., 1., 4., 9.]))

In [56]:
# Clear the previous gradients stored in x to ensure a fresh calculation
x.grad.zero_()

# Perform backpropagation on y = x * x
# Since y is still connected to x in the graph, the derivative is dy/dx = 2x
y.sum().backward()

# Verification:
# This will return True (or a tensor of True values) because 
# the gradient of x^2 is indeed 2*x.
x.grad == 2 * x

tensor([True, True, True, True])

In [14]:
def f(a):
    # Initial operation: b is twice a
    b = a * 2
    
    # Dynamic Loop: The number of iterations depends on the value of 'b'.
    # PyTorch records each multiplication in the graph during execution.
    while b.norm() < 1000:
        b = b * 2
        
    # Control Flow: The path taken depends on the data. 
    # Autograd records which branch was taken (the 'if' or the 'else').
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
        
    return c

In [17]:
# Create a random scalar tensor 'a' and enable gradient tracking
a = torch.randn(size=(), requires_grad=True)

# Pass 'a' through the dynamic function. 
# PyTorch builds the graph on-the-fly based on how many 
# loop iterations occur and which 'if' branch is taken.
d = f(a)

# Backpropagate to calculate the gradient da/dd
d.backward()

In [18]:
a.grad == d / a

tensor(True)